# Using biotrainer autoeval for plm evaluation

This notebook shows an example how to use the biotrainer `autoeval` module for automatic plm evaluation. We use the [PBC](https://github.com/Rostlab/pbc) framework that includes curated datasets that are established for plm benchmarking.
All configs used for evaluating the model on the respective tasks can be found [here](../../biotrainer/autoeval/pbc/pbc_config_bank.py).

In [ ]:
# Install biotrainer if you haven't
# !pip install biotrainer

## Default Use Case: Model Download from Huggingface

The most convient option to use the biotrainer autoeval pipeline is to use the huggingface id of your plm. This will automatically download the model, calculate the embeddings and run the evaluation.

In [ ]:
from typing import List, Generator, Tuple

# Define variables
from biotrainer.autoeval import AutoEval, AvailableFramework

embedder_name = "Rostlab/prot_t5_xl_uniref50"  # Replace with your plm's huggingface id. For alternatives, see "Advanced options" below
min_seq_length = 0  # Default
max_seq_length = 2000  # Default

In [ ]:
# Construct the pipeline
autoeval_pipeline = AutoEval(embedder_name=embedder_name, min_seq_length=min_seq_length, max_seq_length=max_seq_length).pbc_supervised()

In [ ]:
# Run the pipeline
autoeval_report = autoeval_pipeline.run()

In [ ]:
# Let's look at the results
autoeval_report.summary()

## Advanced options 1: Using a custom embedding function

If you are running biotrainer-autoeval directly after training your model, the model will probably not be available on huggingface, but locally. Therefore, you can provide a custom embedder class to be independent of the biotrainer embedding module. The class functions (per-sequence, per-residue) each take a list of strings (sequences) as input and must return, for each sequence, the sequence and the respective embedding. This is to ensure that the sequence is always mapped to the correct embedding.

*What is a generator function?*

A generator function returns a result as soon as it is available, and only continues to create new results after the previous one has been processed. In this case, this is useful because it allows to save the embeddings after computation, thus avoiding that the RAM runs full with the embeddings.

In [ ]:
# Abstract Explanation - Custom Embedder Interface here as a reference
from biotrainer.embedding import CustomEmbedder

# class CustomEmbedder:
#     """ Custom Embedder Interface - Used to provide a standardized interface for autoeval custom embedders """
#     def per_residue(self, seqs: List[str]) -> Generator[Tuple[str, torch.Tensor], None, None]:
#         """ Embed a list of sequences and yield a tuple of (sequence, per-residue embedding for this sequence) """
#         pass
#
#     def per_sequence(self, seqs: List[str]) -> Generator[Tuple[str, torch.Tensor], None, None]:
#         """ Embed a list of sequences and yield a tuple of (sequence, per-sequence embedding for this sequence) """
#         pass

### Concrete Example with the ProtT5 QuickStart Tutorial

Now we use the [ProtT5 QuickStart Tutorial](https://github.com/agemagician/ProtTrans?tab=readme-ov-file#-quick-start) to show how to implement ProtT5 via the custom_embedding_functions into autoeval. Note that this example does not use batching efficiently, but it is a good starting point for your own implementation:

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import re

device = 'cpu' #torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load the tokenizer
tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc', do_lower_case=False)

# Load the model
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc").to(device)

# only GPUs support half-precision currently; if you want to run on CPU use full-precision (not recommended, much slower)
if device == torch.device("cpu"):
    model.to(torch.float32)

class ProtT5CustomEmbedder(CustomEmbedder):
    def per_residue(self, sequences: List[str]) -> Generator[Tuple[str, torch.Tensor], None, None]:
        for sequence in sequences:
            # replace all rare/ambiguous amino acids by X and introduce white-space between all amino acids
            sequence_cleaned = [" ".join(list(re.sub(r"[UZOB]", "X", sequence)))]
            ids = tokenizer(sequence_cleaned, add_special_tokens=True, padding="longest")

            input_ids = torch.tensor(ids['input_ids']).to(device)
            attention_mask = torch.tensor(ids['attention_mask']).to(device)
            # generate embeddings
            with torch.no_grad():
                embedding_repr = model(input_ids=input_ids, attention_mask=attention_mask)
            embedding = embedding_repr.last_hidden_state[0,:len(sequence)]
            yield sequence, embedding

    def per_sequence(self, sequences: List[str]) -> Generator[Tuple[str, torch.Tensor], None, None]:
        for seq, embedding in self.per_residue(sequences):
            yield seq, embedding.mean(dim=0) # shape (1024)


# Run Autoeval
autoeval_prott5custom_pipeline = AutoEval(embedder_name="ProtT5-custom", min_seq_length=min_seq_length, max_seq_length=max_seq_length,
                                          custom_embedder=ProtT5CustomEmbedder()).pbc_supervised()
prott5_report = autoeval_prott5custom_pipeline.run()

## Advanced Options 2: Precomputed embeddings file

Another option is to use precomputed embeddings file, if you prefer that or have them already. Just make sure that the files include embeddings for all framework sequences and are stored by sequence hash, according to biotrainer standards.

In [ ]:
from pathlib import Path
from biotrainer.autoeval import get_unique_framework_sequences

_, per_residue_seqs, per_sequence_seqs = get_unique_framework_sequences(framework=AvailableFramework.PBC_SUPERVISED,
                                                                        min_seq_length=min_seq_length,
                                                                        max_seq_length=max_seq_length)
# per_residue_seqs and per_sequence_seqs are dictionaries mapping sequence hashes to SequenceData objects, use that hash as an id when storing your embeddings

per_residue_path = Path()  # TODO Your per-residue embeddings path
per_sequence_path = Path()  # TODO Your per-sequence embeddings path
report = AutoEval(embedder_name=embedder_name, precomputed_per_residue_embeddings=per_residue_path, precomputed_per_sequence_embeddings=per_sequence_path, min_seq_length=min_seq_length, max_seq_length=max_seq_length).pbc_supervised().run()
report.summary()